In [2]:
# cell 1
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import xgboost as xgb  # Moved up from Cell 10/14

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.cluster import KMeans  # Moved up from Cell 8

# --- 1. GLOBAL CONFIGURATION ---
class CONFIG:
    SEED = 42
    SPLIT_RATIO = 0.90  # From Cell 2
    
    # Path Configuration
    if os.path.exists('/kaggle/input'):
        DATA_PATH = '/kaggle/input/trademaster-cup-2025/'
    else:
        DATA_PATH = '../data/raw'
    
    # XGBoost Hyperparameters (From Cell 14 - Champion Model)
    XGB_PARAMS = {
        'objective': 'reg:absoluteerror',
        'tree_method': 'hist',
        'learning_rate': 0.03,
        'max_depth': 6,
        'subsample': 0.7,
        'colsample_bytree': 0.7,
        'n_jobs': -1,
        'verbosity': 0,
        'random_state': 42
    }

# --- 2. DETERMINISTIC SEEDING ---
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CONFIG.SEED)

# --- 3. DEVICE DETECTION ---
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("✅ Using Apple MPS (Metal Performance Shaders) Acceleration")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("✅ Using NVIDIA CUDA Acceleration")
else:
    DEVICE = torch.device("cpu")
    print("⚠️ Using CPU (Might be slow)")

print(f"🔧 Global Configuration Loaded. Data Path: {CONFIG.DATA_PATH}")

✅ Using Apple MPS (Metal Performance Shaders) Acceleration
🔧 Global Configuration Loaded. Data Path: ../data/raw


In [3]:
# cell 2
# --- DATA LOADING & INITIALIZATION ---
print("🚀 Loading Data...")

# 1. Load & Sort (Merging logic from old Cell 2 & 3)
train_df = pd.read_csv(os.path.join(CONFIG.DATA_PATH, 'train_v2.csv')).sort_values(['date_id', 'minute_id'])
test_df = pd.read_csv(os.path.join(CONFIG.DATA_PATH, 'test_v2.csv')).sort_values(['date_id', 'minute_id'])

# 2. Apply Global Split (Derived from CONFIG)
# We calculate the integer index here because it depends on the data length
SPLIT_IDX = int(len(train_df) * CONFIG.SPLIT_RATIO)
VAL_IDS = train_df['id'].iloc[SPLIT_IDX:].values

# 3. Define Column Groups (Global variables for the pipeline)
target_cols = ['target_short', 'target_medium', 'target_long']
ignore_cols = ['id', 'date_id', 'minute_id', 'row_id'] + target_cols

# Dynamic feature list (everything that isn't ID or Target)
base_features = [c for c in train_df.columns if c not in ignore_cols]

# The "VIP" list from your original Cell 3
vip_features = ['feature_19', 'feature_5', 'feature_27', 'feature_2', 'feature_13']

print(f"✅ Data Loaded. Train: {train_df.shape}, Test: {test_df.shape}")
print(f"🔒 Global Split Index: {SPLIT_IDX} (Ratio: {CONFIG.SPLIT_RATIO})")
print(f"🔒 Validation Rows: {len(VAL_IDS)}")

🚀 Loading Data...
✅ Data Loaded. Train: (139392, 37), Test: (34348, 34)
🔒 Global Split Index: 125452 (Ratio: 0.9)
🔒 Validation Rows: 13940


In [4]:
# cell 3
# --- THE SNIPER ENGINE (Feature Engineering & Scaling) ---
print("🚀 Starting Sniper Data Pipeline (Safe Mode)...")

# --- A. FEATURE ENGINEERING FUNCTIONS ---

def create_lags_fast(df, features, lags=[1, 2, 3, 5]):
    print(f"   Generating Lags...")
    new_cols = []
    for col in features:
        for lag in lags:
            s = df.groupby('date_id')[col].shift(lag)
            s.name = f'{col}_lag{lag}'
            new_cols.append(s)
    return pd.concat([df] + new_cols, axis=1)

def create_vip_features(df):
    print(f"   Generating VIP Interactions & Rolling Stats...")
    new_cols = []
    windows = [5, 10, 20]
    for col in vip_features:
        for w in windows:
            s_mean = df.groupby('date_id')[col].transform(lambda x: x.rolling(w).mean())
            s_mean.name = f'{col}_mean_{w}'
            new_cols.append(s_mean)
            s_std = df.groupby('date_id')[col].transform(lambda x: x.rolling(w).std())
            s_std.name = f'{col}_std_{w}'
            new_cols.append(s_std)

    f19 = df['feature_19']
    for col in ['feature_5', 'feature_27']:
        s_mult = f19 * df[col]
        s_mult.name = f'feat_19_x_{col}'
        new_cols.append(s_mult)
    return pd.concat([df] + new_cols, axis=1)

def create_rank_features(df, vip_cols):
    df_eng = df.copy()
    print(f"   Generating Rolling Ranks for {len(vip_cols)} VIPs...")
    for col in vip_cols:
        df_eng[f'{col}_rank'] = df_eng.groupby('date_id')[col].transform(
            lambda x: x.rolling(window=60, min_periods=10).rank(pct=True)
        )
    return df_eng

def create_delta_features(df, vip_cols):
    df_eng = df.copy()
    print("   Generating Velocity Deltas...")
    for col in vip_cols:
        if f'{col}_lag1' in df_eng.columns:
            df_eng[f'{col}_delta'] = df_eng[col] - df_eng[f'{col}_lag1']
    return df_eng

def create_market_features(df, vip_cols):
    df_eng = df.copy()
    print(f"   Generating Global Context for {len(vip_cols)} VIPs...")
    for col in vip_cols:
        market_mean = df_eng.groupby('date_id')[col].transform(lambda x: x.expanding().mean())
        df_eng[f'global_mean_{col}'] = market_mean
        df_eng[f'divergence_{col}'] = df_eng[col] - market_mean
        df_eng[f'global_std_{col}'] = df_eng.groupby('date_id')[col].transform(lambda x: x.expanding().std())
    return df_eng

def create_intraday_features(df, vip_cols):
    df_eng = df.copy()
    print("   Generating Clock & Pseudo-Price Features...")
    df_eng['dist_from_open'] = df_eng['minute_id']
    max_min = df_eng['minute_id'].max()
    df_eng['dist_from_close'] = max_min - df_eng['minute_id']
    
    for col in vip_cols:
        df_eng[f'cum_{col}'] = df_eng.groupby('date_id')[col].cumsum()
        day_max = df_eng.groupby('date_id')[col].cummax()
        day_min = df_eng.groupby('date_id')[col].cummin()
        range_vals = day_max - day_min
        range_vals = np.where(range_vals == 0, 1, range_vals)
        df_eng[f'day_position_{col}'] = (df_eng[col] - day_min) / range_vals
    return df_eng

# --- B. EXECUTE PIPELINE ---

# 1. Base Engineering (Lags + VIPs)
train_eng = create_vip_features(create_lags_fast(train_df, base_features))
test_eng = create_vip_features(create_lags_fast(test_df, base_features))

# 2. Injection (Ranks, Deltas, Market, Intraday)
# We apply these sequentially to build up the feature set
for df_curr in [train_eng, test_eng]:
    # Note: We are modifying the DataFrame variable in the loop, but since we need to 
    # re-assign to train_eng/test_eng, we do it explicitly below to be safe.
    pass 

# Explicit application to ensure state is saved correctly
vips = ['feature_19', 'feature_5', 'feature_27', 'feature_2', 'feature_13']
vips_context = ['feature_19', 'feature_5', 'feature_27']

train_eng = create_intraday_features(
    create_market_features(
        create_delta_features(
            create_rank_features(train_eng, vips), vips
        ), vips_context
    ), vips_context
)

test_eng = create_intraday_features(
    create_market_features(
        create_delta_features(
            create_rank_features(test_eng, vips), vips
        ), vips_context
    ), vips_context
)

# --- C. PREPARE ARRAYS & CLEANING ---
final_features = [c for c in train_eng.columns if c not in ignore_cols]
print(f"✅ Final Feature Count: {len(final_features)}")

X = train_eng[final_features].values
y = train_eng[target_cols].values
X_test = test_eng[final_features].values

# Initial Sanitization (NaNs to 0)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)

# --- D. STRICT SPLIT & SCALE (Replacing Cell 7 Logic) ---
# We use the SPLIT_IDX calculated in Cell 2
print(f"   Splitting {len(X)} rows at index {SPLIT_IDX}...")

X_tr_raw = X[:SPLIT_IDX]
X_val_raw = X[SPLIT_IDX:]
y_tr = y[:SPLIT_IDX]
y_val = y[SPLIT_IDX:]

print("   🛡️ Fitting RobustScaler on TRAIN set only...")
scaler = RobustScaler()
X_tr_scaled = scaler.fit_transform(X_tr_raw)  # Fit on Train
X_val_scaled = scaler.transform(X_val_raw)    # Transform Val
X_test_scaled = scaler.transform(X_test)      # Transform Test

# Final Sanitization after scaling
X_tr_scaled = np.nan_to_num(X_tr_scaled, nan=0.0)
X_val_scaled = np.nan_to_num(X_val_scaled, nan=0.0)
X_test_scaled = np.nan_to_num(X_test_scaled, nan=0.0)

print(f"✅ SNIPER PIPELINE COMPLETE.")
print(f"   X_tr_scaled shape: {X_tr_scaled.shape}")
print(f"   X_val_scaled shape: {X_val_scaled.shape}")

🚀 Starting Sniper Data Pipeline (Safe Mode)...
   Generating Lags...
   Generating VIP Interactions & Rolling Stats...
   Generating Lags...
   Generating VIP Interactions & Rolling Stats...
   Generating Rolling Ranks for 5 VIPs...
   Generating Velocity Deltas...
   Generating Global Context for 3 VIPs...
   Generating Clock & Pseudo-Price Features...
   Generating Rolling Ranks for 5 VIPs...
   Generating Velocity Deltas...
   Generating Global Context for 3 VIPs...
   Generating Clock & Pseudo-Price Features...
✅ Final Feature Count: 214
   Splitting 139392 rows at index 125452...
   🛡️ Fitting RobustScaler on TRAIN set only...
✅ SNIPER PIPELINE COMPLETE.
   X_tr_scaled shape: (125452, 214)
   X_val_scaled shape: (13940, 214)


In [5]:
# cell 4
# --- THE REFINERY (Clustering + Smart Feature Injection) ---
print("⚖️ REFINERY: Starting Feature Engineering (Smart Mode)...")

# --- PART 1: CLUSTERING (Logic from Old Cell 8) ---
print("🚀 Generating Cluster Features (Honest Mode)...")

# 1. Fit KMeans on TRAIN Only (with subsampling for speed, as in original)
kmeans = KMeans(n_clusters=7, random_state=CONFIG.SEED, n_init=10)
kmeans.fit(X_tr_scaled[::10]) 

# 2. Predict Clusters (Get Labels)
tr_clusters = kmeans.predict(X_tr_scaled)
val_clusters = kmeans.predict(X_val_scaled)
test_clusters = kmeans.predict(X_test_scaled)

# 3. Create One-Hot Encoded Features (The "Tree" Dataset Base)
def add_clusters(X_data, clusters):
    oh = np.eye(7)[clusters]
    return np.hstack([X_data, oh])

X_tr_clustered = add_clusters(X_tr_scaled, tr_clusters)
X_val_clustered = add_clusters(X_val_scaled, val_clusters)
X_test_clustered = add_clusters(X_test_scaled, test_clusters)

print(f"✅ Clusters Added. Intermediate Shape: {X_tr_clustered.shape}")


# --- PART 2: REFINERY & SCOUT (Logic from Old Cell 13) ---
print("   🦅 Training Scout to identify Top Features...")

# 1. Train Scout Model (to find what matters)
# We use target_medium (index 1) as the proxy for general signal, preserving original logic
dtrain_scout = xgb.DMatrix(X_tr_clustered, label=y_tr[:, 1]) 
model_scout = xgb.train({'tree_method':'hist', 'max_depth':4, 'random_state':CONFIG.SEED}, dtrain_scout, num_boost_round=100)
scores = model_scout.get_score(importance_type='total_gain')

# 2. Identify Top Features
sorted_feats = sorted(scores, key=scores.get, reverse=True)
top_10_indices = [int(f[1:]) for f in sorted_feats[:10]] # Extract integer index from 'f123'
top_3_indices = [int(f[1:]) for f in sorted_feats[:3]]
best_feat_idx = top_3_indices[0]

print(f"      👉 Top 10 Features: {top_10_indices}")

# 3. Generate "Cluster Deltas" (Smart)
print("   ➖ Generating Cluster Deltas (Using Top 10 Features)...")

def get_cluster_stats(X_data, cluster_ids_train, target_indices):
    stats = {}
    for feat_idx in target_indices:
        feat_stats = []
        for c in range(7): 
            mask = (cluster_ids_train == c)
            # Safe mean calculation
            mean_val = X_data[mask, feat_idx].mean() if mask.sum() > 0 else 0
            feat_stats.append(mean_val)
        stats[feat_idx] = feat_stats
    return stats

def apply_cluster_deltas(X_data, cluster_ids, stats, target_indices):
    new_feats = []
    for feat_idx in target_indices:
        means = stats[feat_idx]
        cluster_means = np.array([means[c] for c in cluster_ids])
        delta = X_data[:, feat_idx] - cluster_means
        new_feats.append(delta.reshape(-1, 1))
    return np.hstack([X_data] + new_feats)

# Calculate stats on TRAIN only
train_stats = get_cluster_stats(X_tr_clustered, tr_clusters, top_10_indices)

# Apply to all
X_tr_new = apply_cluster_deltas(X_tr_clustered, tr_clusters, train_stats, top_10_indices)
X_val_new = apply_cluster_deltas(X_val_clustered, val_clusters, train_stats, top_10_indices)
X_test_new = apply_cluster_deltas(X_test_clustered, test_clusters, train_stats, top_10_indices)

# 4. Kingmaker Interactions
print("   👑 Generating Kingmaker Interactions...")

def add_king_interactions(X_data, king_idx, other_idxs):
    new_feats = []
    king_col = X_data[:, king_idx]
    for idx in other_idxs:
        interact = king_col * X_data[:, idx]
        new_feats.append(interact.reshape(-1, 1))
    return np.hstack([X_data] + new_feats)

# Final Feature Sets
X_tr_refinery = add_king_interactions(X_tr_new, best_feat_idx, top_3_indices)
X_val_refinery = add_king_interactions(X_val_new, best_feat_idx, top_3_indices)
X_test_refinery = add_king_interactions(X_test_new, best_feat_idx, top_3_indices)

print(f"✅ Refinery Features Ready. Final Shape: {X_tr_refinery.shape}")

⚖️ REFINERY: Starting Feature Engineering (Smart Mode)...
🚀 Generating Cluster Features (Honest Mode)...
✅ Clusters Added. Intermediate Shape: (125452, 221)
   🦅 Training Scout to identify Top Features...
      👉 Top 10 Features: [27, 208, 186, 19, 210, 204, 202, 168, 11, 200]
   ➖ Generating Cluster Deltas (Using Top 10 Features)...
   👑 Generating Kingmaker Interactions...
✅ Refinery Features Ready. Final Shape: (125452, 234)


In [6]:
# cell 5
# --- CHAMPION MODEL TRAINING (Chained XGBoost) ---
from datetime import datetime

print("🦁 Training Champion XGBoost (Legal Features) [NaN-Safe Mode]...")

# 1. Setup Data Pointers (Using variables from Cell 4)
X_tr_curr = X_tr_refinery.copy()
X_val_curr = X_val_refinery.copy()
X_test_curr = X_test_refinery.copy()

# 2. Training Loop
final_preds = np.zeros((len(X_test_curr), 3))
cv_scores = []  # Store scores to calculate filename later

for i in range(3):
    print(f"   🔗 Target {i}...")
    
    # A. Honest History Generation (NaN-Safe)
    honest_feat = None
    if i < 2:
        honest_feat = np.full(len(X_tr_curr), np.nan)
        tscv = TimeSeriesSplit(n_splits=5)
        # We only need a light model for the history feature
        for t_idx, v_idx in tscv.split(X_tr_curr):
            m = xgb.train(CONFIG.XGB_PARAMS, xgb.DMatrix(X_tr_curr[t_idx], y_tr[t_idx, i]), num_boost_round=100)
            honest_feat[v_idx] = m.predict(xgb.DMatrix(X_tr_curr[v_idx]))

    # B. Main Model Training
    dtrain = xgb.DMatrix(X_tr_curr, label=y_tr[:, i])
    dval = xgb.DMatrix(X_val_curr, label=y_val[:, i])
    
    # Train with early stopping
    model = xgb.train(CONFIG.XGB_PARAMS, dtrain, num_boost_round=2000, 
                      evals=[(dval, 'val')], early_stopping_rounds=50, verbose_eval=False)
    
    # Store Score (MAE)
    score = model.best_score
    cv_scores.append(score)
    print(f"      ✅ Target {i} Best Score: {score:.6f}")
    
    # C. Prediction & Chaining
    test_pred = model.predict(xgb.DMatrix(X_test_curr))
    val_pred = model.predict(dval)
    final_preds[:, i] = test_pred

    # Add predictions as features for the next target (Chaining)
    if i < 2:
        X_tr_curr = np.hstack([X_tr_curr, honest_feat.reshape(-1, 1)])
        X_val_curr = np.hstack([X_val_curr, val_pred.reshape(-1, 1)])
        X_test_curr = np.hstack([X_test_curr, test_pred.reshape(-1, 1)])

# 3. Submission Generation
sub = pd.DataFrame({'id': test_df['id']})
sub['target_short'] = final_preds[:, 0]
sub['target_medium'] = final_preds[:, 1]
sub['target_long'] = final_preds[:, 2]

# Center the predictions (as in original)
for c in sub.columns[1:]:
    sub[c] = sub[c] - sub[c].mean()

# 4. Dynamic Filename Generation
# Logic: Name + CV*100 + Timestamp

# Weighted MAE: Short(0.5) + Medium(0.3) + Long(0.2)
# Assumes cv_scores is [Short, Medium, Long] which matches the loop order
wmae = (cv_scores[0] * 0.5 + cv_scores[1] * 0.3 + cv_scores[2] * 0.2)
avg_cv = wmae * 100

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"submission_XGB_Refinery_CV{avg_cv:.5f}_{timestamp}.csv"
output_path = f"../submissions/{filename}"

# Ensure directory exists
os.makedirs('../submissions', exist_ok=True)
sub.to_csv(output_path, index=False)

print(f"🚀 SAVED: {output_path}")

🦁 Training Champion XGBoost (Legal Features) [NaN-Safe Mode]...


   🔗 Target 0...
      ✅ Target 0 Best Score: 0.002861
   🔗 Target 1...
      ✅ Target 1 Best Score: 0.007255
   🔗 Target 2...
      ✅ Target 2 Best Score: 0.015620
🚀 SAVED: ../submissions/submission_XGB_Refinery_CV0.67309_20260211_143213.csv


In [7]:
# cell 6
# --- ALTERNATIVE MODELS (For Ensemble Diversity) ---
from datetime import datetime
import xgboost as xgb
import numpy as np
import pandas as pd

print("🧪 Training Alternative Models (Purist & Robust)...")

# --- MODEL A: THE PURIST (Standard Features Only) ---
# This model sees ONLY the raw scaled features (from Cell 3).
# Strategy: Shallower trees (depth=4) to prevent overfitting on raw data.
print("   🌲 Training 'Purist' Model (Base Features Only)...")

X_tr_purist = X_tr_scaled.copy()
X_val_purist = X_val_scaled.copy()
X_test_purist = X_test_scaled.copy()

final_preds_purist = np.zeros((len(X_test_purist), 3))
cv_scores_purist = []

for i in range(3):
    dtrain = xgb.DMatrix(X_tr_purist, label=y_tr[:, i])
    dval = xgb.DMatrix(X_val_purist, label=y_val[:, i])
    
    # Diversity Strategy: Shallower trees than Champion
    params_purist = CONFIG.XGB_PARAMS.copy()
    params_purist['max_depth'] = 4 
    
    model = xgb.train(params_purist, dtrain, num_boost_round=1500, 
                      evals=[(dval, 'val')], early_stopping_rounds=50, verbose_eval=False)
    
    cv_scores_purist.append(model.best_score)
    print(f"      ✅ Target {i} Best Score: {model.best_score:.6f}")
    final_preds_purist[:, i] = model.predict(xgb.DMatrix(X_test_purist))

# Save Purist Submission
sub_purist = pd.DataFrame({'id': test_df['id']})
sub_purist['target_short'] = final_preds_purist[:, 0]
sub_purist['target_medium'] = final_preds_purist[:, 1]
sub_purist['target_long'] = final_preds_purist[:, 2]

for c in sub_purist.columns[1:]:
    sub_purist[c] = sub_purist[c] - sub_purist[c].mean()

# Weighted MAE: Short(0.5) + Medium(0.3) + Long(0.2)
# Assumes cv_scores is [Short, Medium, Long] which matches the loop order
# CORRECT (Uses Purist data)
wmae = (cv_scores_purist[0] * 0.5 + cv_scores_purist[1] * 0.3 + cv_scores_purist[2] * 0.2)
avg_cv_p = wmae * 100

filename_p = f"submission_XGB_Purist_CV{avg_cv_p:.5f}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
output_p = f"../submissions/{filename_p}"
sub_purist.to_csv(output_p, index=False)
print(f"      🚀 Saved Purist: {filename_p}")


# --- MODEL B: THE ROBUST (NaN & Outlier Heavy) ---
# This model uses the FULL Refinery feature set (from Cell 4).
# Strategy: Deeper trees (depth=8) and lower LR to capture non-linear noise.
print("   🛡️ Training 'Robust' Model (Deep Trees)...")

X_tr_rob = X_tr_refinery.copy()
X_val_rob = X_val_refinery.copy()
X_test_rob = X_test_refinery.copy()

final_preds_rob = np.zeros((len(X_test_rob), 3))
cv_scores_rob = []

# Diversity Strategy: Aggressive parameters
params_rob = CONFIG.XGB_PARAMS.copy()
params_rob['max_depth'] = 8         # Deeper
params_rob['learning_rate'] = 0.01  # Slower
params_rob['subsample'] = 0.6       # More randomness

for i in range(3):
    dtrain = xgb.DMatrix(X_tr_rob, label=y_tr[:, i])
    dval = xgb.DMatrix(X_val_rob, label=y_val[:, i])
    
    model = xgb.train(params_rob, dtrain, num_boost_round=3000, 
                      evals=[(dval, 'val')], early_stopping_rounds=100, verbose_eval=False)
    
    cv_scores_rob.append(model.best_score)
    print(f"      ✅ Target {i} Best Score: {model.best_score:.6f}")
    final_preds_rob[:, i] = model.predict(xgb.DMatrix(X_test_rob))

# Save Robust Submission
sub_rob = pd.DataFrame({'id': test_df['id']})
sub_rob['target_short'] = final_preds_rob[:, 0]
sub_rob['target_medium'] = final_preds_rob[:, 1]
sub_rob['target_long'] = final_preds_rob[:, 2]

for c in sub_rob.columns[1:]:
    sub_rob[c] = sub_rob[c] - sub_rob[c].mean()

# Weighted MAE: Short(0.5) + Medium(0.3) + Long(0.2)
# Assumes cv_scores is [Short, Medium, Long] which matches the loop order
# CORRECT (Uses Robust data)
wmae = (cv_scores_rob[0] * 0.5 + cv_scores_rob[1] * 0.3 + cv_scores_rob[2] * 0.2)
avg_cv_r = wmae * 100

filename_r = f"submission_XGB_Robust_CV{avg_cv_r:.5f}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
output_r = f"../submissions/{filename_r}"
sub_rob.to_csv(output_r, index=False)
print(f"      🚀 Saved Robust: {filename_r}")

🧪 Training Alternative Models (Purist & Robust)...
   🌲 Training 'Purist' Model (Base Features Only)...
      ✅ Target 0 Best Score: 0.002856
      ✅ Target 1 Best Score: 0.007285
      ✅ Target 2 Best Score: 0.015425
      🚀 Saved Purist: submission_XGB_Purist_CV0.66987_20260211_143221.csv
   🛡️ Training 'Robust' Model (Deep Trees)...
      ✅ Target 0 Best Score: 0.002861
      ✅ Target 1 Best Score: 0.007272
      ✅ Target 2 Best Score: 0.015458
      🚀 Saved Robust: submission_XGB_Robust_CV0.67037_20260211_143243.csv
